# Correctness-First LLM Router

Thin orchestrator over `src/`. Predicts expected per-model performance, routes by
`argmax(0.85*p_hat - 0.15*c_bar/denom + bias)` with OOF-tuned per-model bias.

See `docs/superpowers/specs/` (design) and `docs/superpowers/plans/` (plan).
Run phases in order; each phase caches artifacts and is independently submittable.

In [ ]:
# Colab / A100 setup. Re-run after changing runtime.
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/NLP Final Project"
!pip -q install "pandas==2.2.2" "numpy>=1.26.4,<2.1" scikit-learn scipy lightgbm joblib tqdm \
    "transformers>=4.48.0" "sentence-transformers>=2.7.0" accelerate
import sys
sys.path.append("/content/drive/MyDrive/NLP Final Project")

## Phase 0 — reframe (TF-IDF + Qwen3 embeddings + handcrafted -> Ridge/LGBM -> calibrate -> bias-tuned routing)
First reuses cached Qwen3-Embedding-4B embeddings if present under `Output/router_a100/cache/`.

In [ ]:
from src.config import CFG
from src.run import run_phase0
res0 = run_phase0(CFG())  # add smoke=True for a fast dry run

## Phase 1 — ModernBERT-large encoder (multilabel + aux difficulty head)
Requires GPU. Caches `enc_oof_*.npy` / `enc_test_*.npy`.

In [ ]:
from src.run import run_phase1
res1 = run_phase1(CFG())  # smoke=True, encoder_epochs=1, encoder_max_len=512 for a dry run

## Phase 2 — vLLM self-consistency / judge difficulty features

In [ ]:
from src.run import run_phase2
res2 = run_phase2(CFG())

## Phase 3 — meta-stacker, bias refinement, final candidate submissions

In [ ]:
from src.run import run_phase3
run_phase3(CFG())